In [1]:
#############################################################################
#### Code to take a 24 band raster and split into 24 rasters with 1 band #### 
#############################################################################

import os
import rasterio

# Change the filepaths here:  *****************************************
input_tif = r"D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\UTCI_20240701_int16.tif" # Filepath to raster to split
output_dir = r"D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\Hourly"         # Filepath to folder to store the split rasters 
#**************8********************************************************

# Create the folder to store rasters if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

with rasterio.open(input_tif) as src:
    meta = src.meta.copy() # keep metadata from the input raster
    meta.update(count=1)  # one band per output file (so change this in metadata)

    for band in range(1, src.count + 1):  # rasterio is 1-indexed
        hour = f"{band:02d}"
        out_name = f"UTCI_20240701_{hour}_int16.tif"
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            # Read band in chunks 
            for _, window in src.block_windows(band):
                data = src.read(band, window=window)
                dst.write(data, 1, window=window)

        print(f"Saved: {out_name}")

print("Raster splitting complete")


Saved: UTCI_20240701_01_int16.tif
Saved: UTCI_20240701_02_int16.tif
Saved: UTCI_20240701_03_int16.tif
Saved: UTCI_20240701_04_int16.tif
Saved: UTCI_20240701_05_int16.tif
Saved: UTCI_20240701_06_int16.tif
Saved: UTCI_20240701_07_int16.tif
Saved: UTCI_20240701_08_int16.tif
Saved: UTCI_20240701_09_int16.tif
Saved: UTCI_20240701_10_int16.tif
Saved: UTCI_20240701_11_int16.tif
Saved: UTCI_20240701_12_int16.tif
Saved: UTCI_20240701_13_int16.tif
Saved: UTCI_20240701_14_int16.tif
Saved: UTCI_20240701_15_int16.tif
Saved: UTCI_20240701_16_int16.tif
Saved: UTCI_20240701_17_int16.tif
Saved: UTCI_20240701_18_int16.tif
Saved: UTCI_20240701_19_int16.tif
Saved: UTCI_20240701_20_int16.tif
Saved: UTCI_20240701_21_int16.tif
Saved: UTCI_20240701_22_int16.tif
Saved: UTCI_20240701_23_int16.tif
Saved: UTCI_20240701_24_int16.tif
Raster splitting complete


In [ ]:
###########################################################################################################
#### Code to take a folder with 24 band rasters (band = hour of day) and create a mean raster per hour #### 
###########################################################################################################

import os
import rasterio
import numpy as np

# Change the filepaths here:  *****************************************
input_dir = r"D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo"                       # Folder with rasters to split
output_dir = r"D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\Hourly_Means"         # Filepath to folder to store the mean rasters 
hour_first = 7    # first hour required
hour_final = 15   # final hour required 
#**********************************************************************

# Create the folder to store rasters if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get a list of all tif files in the folder 
tif_files = [os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.lower().endswith(".tif")]

print(f"{len(tif_files)} rasters in total")

nodata_val = -9999   # can change this if you would like! 

# Get metadata from the first raster 
with rasterio.open(tif_files[0]) as ref:
    meta = ref.meta.copy()
    meta.update(count=1, dtype="int16", nodata=nodata_val)  # output is going to be single-band
    profile = ref.profile


# Loop over hours
for hour in range(hour_first, hour_final+1):

    print(f"\nStarting processing for hour {hour:02d}")

    # Output path
    out_path = os.path.join(output_dir, f"UTCI_hourly_mean_{hour:02d}.tif")

    # Use first raster to define windows
    with rasterio.open(tif_files[0]) as ref:
        windows = list(ref.block_windows(1))

    with rasterio.open(out_path, "w", **meta) as dst:

        for _, window in windows:
            stack = []

            # Read same window from same band (hour) across all rasters
            for tif in tif_files:
                with rasterio.open(tif) as src:
                    band_data = src.read(hour, window=window).astype("float32")

                    # Handle nodata, (raster had no data outside city shape)
                    if src.nodata is not None:
                        band_data = np.where(band_data == src.nodata, np.nan, band_data)

                    stack.append(band_data)

            stack = np.stack(stack, axis=0)

            # Mean ignoring nodata
            mean_data = np.nanmean(stack, axis=0)

            # Convert back to original dtype (should be int16, but could change), use -9999 as the no data value
            mean_data = np.rint(mean_data)  # round to nearest integer, can't store decimals as int16
            mean_data = np.where(np.isnan(mean_data), nodata_val, mean_data)  # set nodata
            mean_data = np.clip(mean_data, -9999, 9999).astype("int16")

            dst.write(mean_data, 1, window=window)

    print(f"Saved: {out_path}")

print("Hourly mean rasters complete")



2 rasters in total

Starting processing for hour 07
Saved: D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\Hourly_Means\UTCI_hourly_mean_07.tif

Starting processing for hour 08
Saved: D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\Hourly_Means\UTCI_hourly_mean_08.tif

Starting processing for hour 09
Saved: D:\DDL\SOLWEIG\SOLWEIG_simulations\1_New_Albedo\Hourly_Means\UTCI_hourly_mean_09.tif

Starting processing for hour 10
